In [1]:
import pandas as pd 
import sqlalchemy as sa
from datetime import datetime, timedelta, timezone

In [6]:
# Proses get connection to database
def connect_to_database(**params):
    username = params.get('username')
    password = params.get('password')
    host = params.get('host')
    port = params.get('port')
    database = params.get('database')
    
    engine = sa.create_engine(
        f"postgresql+psycopg://{username}:{password}@{host}:{port}/{database}"
    )
    connection = engine.connect()

    return(engine, connection)

# Proses ekstraksi data di database
def extract_data_from_database(query, conn):
    data = pd.read_sql(query, conn)
    return(data)

# Proses menambahkan metadata
def add_metadata(data):
    # Get date now
    gmt_plus_7 = timezone(timedelta(hours=14))
    current_time = datetime.now(gmt_plus_7)

    # Tambahkan kolom created date dan isikan dengan tanggal dan waktu hari ini
    data['created_date'] = current_time
    data['created_by'] = 'SYSTEM'
    return(data)

# Proses load data / menyimpan data ke database
def load_data(data, table_name, connection):
    data.to_sql(
        table_name,
        connection,
        if_exists = 'replace',
        index = False
    )

In [8]:
query = """
WITH tbl_order AS (
	SELECT
		order_id,
		customer_id,
		order_maker_id,
		TO_DATE(order_date, 'MM/DD/YYYY') AS order_date,
		order_time,
		completion_time,
		completion_time::TIME - order_time::TIME AS finish_time,
		is_complain,
		complain_detail
	FROM trx_order
),
tbl_order_detail AS (
	SELECT DISTINCT
		order_details_id AS order_detail_id,
		order_id,
		pizza_id,
		quantity
	FROM trx_order_detail
	WHERE order_details_id IS NOT NULL
),
tbl_pizza AS (
	SELECT DISTINCT
		pizza_id,
		pizza_type_id,
		size,
		REPLACE(price, 'IDR', '')::INT AS price,
		REPLACE(production_cost, 'IDR', '')::INT AS production_cost
	FROM mst_pizza
),
tbl_pizza_type AS (
	SELECT
		pizza_type_id,
		name AS pizza_name,
		UPPER(category) AS pizza_category,
		ingredients
	FROM mst_pizza_type
),
tbl_customer AS (
	SELECT DISTINCT
		customer_id,
		customer_name,
		gender AS customer_gender
	FROM mst_customer
)
	SELECT
		o.order_id,
		od.order_detail_id,
		o.customer_id,
		o.order_maker_id,
		c.customer_name,
		c.customer_gender,
		od.pizza_id,
		pt.pizza_name,
		pt.pizza_category,
		o.order_date,
		o.order_time,
		o.completion_time,
		o.finish_time,
		od.quantity,
		pz.price,
		pz.production_cost,
		od.quantity * pz.price AS total_price,
		od.quantity * (pz.price - pz.production_cost) AS total_profit,
		o.is_complain,
		o.complain_detail
	FROM tbl_order o
	INNER JOIN tbl_order_detail od ON o.order_id = od.order_id
	LEFT JOIN tbl_pizza pz ON od.pizza_id = pz.pizza_id
	LEFT JOIN tbl_pizza_type pt ON pt.pizza_type_id = pz.pizza_type_id
	LEFT JOIN tbl_customer c ON o.customer_id = c.customer_id;
"""

# Koneksi ke db_staging (bronze layer)
engine_staging, con_staging = connect_to_database(
    username = 'postgres',
    password = '',
    host = 'localhost',
    port = '5432',
    database = 'db_pizza_staging'
)

In [10]:
# Proses ekstrak data
data_extracted = extract_data_from_database(query, con_staging)
data_extracted = add_metadata(data_extracted)

In [11]:
# Koneksi ke db_mart (golden layer)
engine_mart, con_mart = connect_to_database(
    username = 'postgres',
    password = '',
    host = 'localhost',
    port = '5432',
    database = 'dm_pizza_transaction'
)

In [12]:
load_data(
    data = data_extracted, 
    table_name = 'f_pizza_detail_transaction', 
    connection = con_mart
)

/var/folders/sx/cv3r86bj29v1hj9bpxqzr2b40000gn/T/ipykernel_35438/2738766252.py:34: UserWarning: the 'timedelta' type is not supported, and will be written as integer values (ns frequency) to the database.
  data.to_sql(
